In [22]:
from box import Box

In [23]:
args = Box({
    'batch_size': 32,
    'cola_classifier_path': '/content/drive/MyDrive/style_transfer/cola_classifier',
    'wieting_tokenizer_path': 'sim.sp.30k.model',
    'wieting_model_path': 'sim.pt',
    't1': 75., # this is default value
    't2': 70., # this is default value
    't3': 12. # this is default value
})

In [24]:
epochs = 3
# ckpt_num = 0
ckpt_num = 1975*3
lr = 2e-4
batch_size = 8
num_unmask_steps = 128
baseline = False

if baseline:
    results_path = f'./results/baseline.jsonl'
    output_path = f'./results/baseline_eval.json'
else:
    if ckpt_num == 0:
        results_path = f'./results/llada_untrained/result_{num_unmask_steps}un.jsonl'
        output_path = f'./results/llada_untrained/eval_{num_unmask_steps}un.json'
    else:
        results_path = f'./results/llada_{epochs}ep-{batch_size}bs-{lr}lr/result_{num_unmask_steps}un_{ckpt_num}ch.jsonl'
        output_path = f'./results/llada_{epochs}ep-{batch_size}bs-{lr}lr/eval_{num_unmask_steps}un_{ckpt_num}ch.json'
print("Results will be saved to:", output_path)

Results will be saved to: ./results/llada_3ep-8bs-0.0002lr/eval_128un_5925ch.json


In [25]:
import json

preds = []
inputs = []
hates = []

with open(results_path, 'r') as f:
	for line in f:
		result = json.loads(line)
		pred = result['pred']
		pred = pred.split('<|endoftext|>')[0].strip()
		preds.append(pred.lower())
		input = result['actual']
		input = input.split('<|endoftext|>')[0].strip()
		inputs.append(input.lower())
		hate = result['hate']
		hates.append(hate.lower())

In [26]:
len(preds), len(inputs)

(198, 198)

In [27]:
import torch

### Style Transfer Accuracy metric

In [28]:
from paradetox.evaluation_detox.metric_tools.style_transfer_accuracy import classify_preds
import numpy as np

In [29]:
accuracy_by_sent = classify_preds(args, preds)
accuracy = np.mean(accuracy_by_sent)
print('\n')
accuracy_by_sent = torch.tensor(accuracy_by_sent)
print(accuracy_by_sent)
print(accuracy)

Calculating style of predictions


Some weights of the model checkpoint at SkolkovoInstitute/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 7/7 [00:02<00:00,  2.95it/s]



tensor([1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1.,
        1., 0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 0., 1., 0., 1., 1., 0., 0., 1., 1., 1., 1., 0.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 0., 1.,
        1., 1., 1., 1., 1., 1., 1., 0., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1.,
        1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
0.898989898989899


### Content Similarity metrics

In [30]:
from paradetox.evaluation_detox.metric_tools.content_similarity import flair_sim

In [31]:
emb_sim_stats = flair_sim(args, inputs, preds)
emb_sim = emb_sim_stats.mean()
print('\n')
print(emb_sim_stats)
print(emb_sim)

Calculating flair embeddings similarity


tensor([0.8129, 1.0000, 0.9567, 0.8224, 1.0000, 0.8615, 0.9663, 0.9590, 0.7139,
        0.8845, 1.0000, 0.8752, 0.8923, 0.8600, 0.8608, 0.8726, 0.7547, 0.9662,
        0.7298, 0.9237, 0.9563, 0.9542, 0.9582, 0.9340, 1.0000, 0.9302, 0.4655,
        0.8403, 0.9690, 1.0000, 1.0000, 1.0000, 1.0000, 0.8787, 0.8873, 0.9757,
        0.9544, 0.7403, 0.9510, 0.6527, 0.8363, 0.9080, 0.8829, 0.9853, 0.9214,
        0.9941, 0.9252, 0.9765, 1.0000, 0.9637, 0.9195, 1.0000, 0.9545, 0.8489,
        0.9659, 0.9916, 0.7013, 0.8807, 0.9726, 0.9523, 0.8767, 0.9456, 0.9118,
        0.9344, 0.9555, 0.3980, 0.9906, 0.7950, 0.7468, 0.9266, 0.9258, 0.8649,
        0.6864, 0.8968, 1.0000, 1.0000, 1.0000, 0.7030, 0.8019, 1.0000, 0.7427,
        0.9370, 1.0000, 0.9203, 0.8793, 0.9368, 0.9313, 0.9834, 0.8732, 0.8335,
        0.9228, 1.0000, 0.9539, 0.8485, 0.7260, 0.9766, 0.9754, 0.9489, 0.8749,
        0.9640, 0.9757, 0.8052, 0.8735, 0.9583, 0.7652, 0.5213, 0.8095, 0.9277

### Fluency metrics

In [32]:
from paradetox.evaluation_detox.metric_tools.fluency import cola_fluency

In [33]:
cola_stats = cola_fluency(preds)
cola_acc = sum(cola_stats) / len(preds)
print('\n')
cola_stats = torch.tensor(cola_stats)
print(cola_stats)
print(cola_acc)



tensor([1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1,
        0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1,
        1, 1, 1, 1, 1, 1])
0.7626262626262627


In [34]:
cola_stats = cola_fluency(preds)
cola_acc = sum(cola_stats) / len(preds)
print('\n')
cola_stats = torch.tensor(cola_stats)
print(cola_stats)
print(cola_acc)



tensor([1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1,
        0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1,
        1, 1, 1, 1, 1, 1])
0.7626262626262627


In [35]:
ours = torch.tensor([1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
        0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0,
        0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0,
        1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1,
        1, 1, 1, 1, 1, 1])

baseline = torch.tensor([1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0,
        1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
        1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0,
        1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1,
        1, 1, 1, 0, 1, 1])



print(torch.where(ours > baseline)[0] + 1)
print(torch.where(ours < baseline)[0] + 1)

tensor([  3,  18,  29,  41,  54,  69,  79,  82, 100, 112, 148, 196])
tensor([  9,  17,  28,  42,  44,  76,  81,  85,  96,  97, 103, 110, 111, 125,
        133, 136, 138, 143, 164, 168, 170, 189, 190])


In [36]:
preds_cola = cola_fluency(preds)
hates_cola = cola_fluency(hates)
print(torch.where(torch.tensor(preds_cola) < torch.tensor(hates_cola))[0] + 1)
print(torch.where(torch.tensor(preds_cola) > torch.tensor(hates_cola))[0] + 1)

tensor([ 42,  72, 110, 125, 143, 170, 185, 187])
tensor([  1,   5,  18,  64,  70,  82,  94,  99, 102, 112, 131, 148, 160, 183,
        194, 196])


### Joint metrics

In [37]:
from paradetox.evaluation_detox.metric_tools.joint_metrics import *

In [38]:
joint = get_j(args, accuracy_by_sent, emb_sim_stats, cola_stats, preds)
print(joint)

tensor(0.6364)


### Saving Results

In [39]:
eval_dict = {
    "STA": float(accuracy),
	"SIM": float(emb_sim),
    "FL": float(cola_acc),
	"J": float(joint)
}

In [40]:
with open(output_path, 'w') as f:
	json.dump(eval_dict, f, indent=4)